# Ajuste de hiperparámetros de CNN con Fashion-MNIST
### TensorFlow + Keras Tuner (Hyperband) · PyTorch Lightning + Optuna (Random Search)

**Objetivos.** Comparar dos métodos de búsqueda con la **misma arquitectura conceptual**, las mismas imágenes y la misma partición de entrenamiento/validación/prueba. 
**Flujo:** datos → CNN parametrizada → búsqueda → mejor configuración → reentrenamiento → evaluación en test.

In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import pandas as pd
print("TensorFlow:", tf.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
## !pip install keras-tuner -q

import keras_tuner as kt

In [ ]:
(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    name="fashion_mnist",
    split=[
        "train[:40%]",
        "train[40%:50%]",
        "test[:50%]"
    ],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = ds_info.features["label"].num_classes
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]

print(ds_info)
print("Número de clases:", NUM_CLASSES)

In [ ]:
fig = tfds.show_examples(ds_train_raw, ds_info)

In [ ]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.int32)

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    return image, label

def make_dataset(raw, training=False):
    ds = raw.map(preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(ds_train_raw, training=True)
val_ds = make_dataset(ds_val_raw)
test_ds = make_dataset(ds_test_raw)
for x_batch, y_batch in train_ds.take(1):
    print('x:', x_batch.shape, x_batch.dtype, '| y:', y_batch.shape, y_batch.dtype)
    print('Rango de píxeles:', float(tf.reduce_min(x_batch)), float(tf.reduce_max(x_batch)))

##  TensorFlow / Keras Tuner: Hyperband
https://keras.io/keras_tuner/api/hyperparameters/#int-method

Hyperband empieza con varios candidatos y asigna más épocas a los más prometedores.

In [ ]:
SEARCH_SPACE = {
    "learning_rate": [1e-2, 1e-3],
    "dropout": [0.5, 0.2],
    "filters": [8, 16],
    "activation": ["relu", "elu"],
}


def build_tf_model(hp):
    
    filters = hp.Choice("filters", SEARCH_SPACE["filters"])
    activation = hp.Choice("activation", SEARCH_SPACE["activation"])
    dropout = hp.Choice("dropout", SEARCH_SPACE["dropout"])
    lr = hp.Choice("learning_rate", SEARCH_SPACE["learning_rate"])

    inputs = tf.keras.Input(shape=(28, 28, 1))
    x = inputs

    n_layers = hp.Int("conv_layers", min_value=1, max_value=3)
    for i in range(n_layers):
        x = tf.keras.layers.Conv2D(
            filters=filters,
            kernel_size=hp.Int(f"kernel_size_{i}", 3, 5, step=2),
            activation=activation,
            padding="same",
            name=f"conv_{i}",
        )(x)

        pool = hp.Choice(f"pooling_{i}", ["max", "avg"])
        Pool = tf.keras.layers.MaxPooling2D if pool == "max" else tf.keras.layers.AveragePooling2D
        x = Pool(pool_size=2, name=f"{pool}pool_{i}")(x)

        x = tf.keras.layers.BatchNormalization(name=f"bn_{i}")(x)
        if hp.Boolean("dropout"):
            x = tf.keras.layers.Dropout(dropout, name=f"drop_{i}")(x)


    gpool = hp.Choice("global_pooling", ["max", "avg"])
    GPool = tf.keras.layers.GlobalMaxPooling2D if gpool == "max" else tf.keras.layers.GlobalAveragePooling2D
    x = GPool(name=f"global_{gpool}")(x)

    outputs = tf.keras.layers.Dense(10, activation="softmax", name="probs")(x)

    model = tf.keras.Model(inputs, outputs, name="cnn_tunable")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
MAX_EPOCHS = 5
tuner = kt.Hyperband(
    build_tf_model, objective='val_accuracy',
    max_epochs=MAX_EPOCHS, factor=3, hyperband_iterations=1,
    directory='tuning_results', project_name='tf_hyperband_fashion',
    overwrite=True, seed=SEED)
tuner.search_space_summary()

In [ ]:
tuner.search(train_ds, validation_data=val_ds,
             batch_size=BATCH_SIZE, verbose=1)
best_hp = tuner.get_best_hyperparameters(1)[0]
best_tf_params = {k: best_hp.get(k) for k in SEARCH_SPACE}
print('Mejor Hyperband:', best_tf_params)

In [ ]:
import pandas as pd
import numpy as np


def trials_to_dataframe(tuner, sort_by="score", ascending=False):

    rows = []
    for trial in tuner.oracle.trials.values():
        hps = trial.hyperparameters.values.copy()

        metrics = {}
        if trial.metrics is not None:
            for name, values in trial.metrics.metrics.items():
                if name.startswith("val_"):
                    metrics[name] = values.get_best_value() 

            for name, values in trial.metrics.metrics.items():
                if not name.startswith("val_"):
                    metrics[name] = values.get_best_value()


        rows.append({
            "status": trial.status,
            "score": trial.score,        
            "best_step": getattr(trial, "best_step", np.nan),
            **hps,
            **metrics,
        })

    df = pd.DataFrame(rows)
    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=ascending, na_position="last")
    return df.reset_index(drop=True)

In [ ]:
trials_to_dataframe(tuner)

In [ ]:
models = tuner.get_best_models(num_models=2)
best_model = models[0]
best_model.summary()

Reentrenamiento de la mejor configuración y evaluación en TEST
Para ambos frameworks se vuelve a entrenar desde cero la configuración seleccionada usando únicamente el subconjunto **train**, vigilando `val`. Después se consulta el test oficial **una sola vez** por implementación.

In [ ]:
history_tf = best_model.fit(
    train_ds, validation_data=val_ds,
    batch_size=BATCH_SIZE, epochs=MAX_EPOCHS, verbose=1)
tf_test_loss, tf_test_acc = best_model.evaluate(test_ds, verbose=1)
print(f'TensorFlow test accuracy: {tf_test_acc:.4f}')

## 3. PyTorch Lightning + Optuna: Random Search
Se utilizan los mismos arrays, la misma forma de CNN, Adam, entropía cruzada y el mismo espacio discreto de cuatro hiperparámetros. `RandomSampler` prueba combinaciones aleatorias **sin utilizar los resultados anteriores para proponerlas**. No activamos pruning para conservar el carácter de Random Search.

https://optuna.org/

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torchvision.datasets import FashionMNIST
import torchvision.transforms as T
from torch.utils.data import random_split
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
SEED = 42

In [ ]:
# pip install optuna
import optuna
from optuna.integration import PyTorchLightningPruningCallback

In [ ]:
base_transform = T.Compose([
    T.ToTensor(),     
])

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),    
    T.ToTensor(),
])

val_transform = base_transform
test_transform = base_transform

In [ ]:
train_ds_full = FashionMNIST(root="./data", train=True,  download=True,
                        transform=train_transform)
test_ds_full  = FashionMNIST(root="./data", train=False, download=True,
                        transform=val_transform)

from torch.utils.data import Subset
N = 1000
train_ds = Subset(train_ds_full, range(N*10))
test_ds = Subset(test_ds_full, range(N))

In [ ]:

val_size   = int(0.1 * len(train_ds))
train_size = len(train_ds) - val_size
train_ds, val_ds = random_split(
    train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

In [ ]:
BATCH_SIZE=32
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False)
x_batch, y_batch = next(iter(train_loader))
print("x:", x_batch.shape, x_batch.dtype, "| y:", y_batch.shape, y_batch.dtype)
print("Rango de píxeles:", float(x_batch.min()), float(x_batch.max()))

In [ ]:
class LitCNN(L.LightningModule):
    def __init__(self, num_classes=10, lr = 1e-3, dropout = 0.3,
                 base_filters=16):
        super().__init__()
        self.save_hyperparameters()

        self.features = nn.Sequential(
            nn.Conv2d(1, base_filters, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),     

            nn.Conv2d(base_filters, base_filters * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_filters * 2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                   

            nn.Conv2d(base_filters * 2, base_filters * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_filters * 4),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),        
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(base_filters * 4, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

    def _step(self, batch, stage):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log(f'{stage}_loss', loss, on_epoch=True, on_step=False,
                 prog_bar=(stage != 'train'), batch_size=x.size(0))
        self.log(f'{stage}_accuracy', acc, on_epoch=True, on_step=False,
                 prog_bar=True, batch_size=x.size(0))
        return loss

    def training_step(self, batch, _):
        return self._step(batch, "train")

    def validation_step(self, batch, _):
        self._step(batch, "val")

    def test_step(self, batch, _):
        self._step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr)

In [ ]:
def objective(trial):
    lr           = trial.suggest_float("lr", 1e-3, 1e-2, log=True)
    dropout      = trial.suggest_float("dropout", 0.0, 0.3, step=0.1)
    base_filters = trial.suggest_categorical("base_filters", [8,16, 32])

    model = LitCNN(lr=lr, dropout=dropout, base_filters=base_filters)

    callbacks = [
        L.pytorch.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=3),
        PyTorchLightningPruningCallback(trial, monitor="val_accuracy"),
    ]

    trainer = L.Trainer(max_epochs=5,callbacks=callbacks)
    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_accuracy"].item()

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.RandomSampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner())

study.optimize(objective, n_trials=5, timeout=None)

print("Best val_accuracy:", study.best_value)
print("Best params :", study.best_params)

In [ ]:
torch_results = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
display(torch_results.sort_values('value', ascending=False))
import numpy as np
fig, ax = plt.subplots(figsize=(8, 4))
scores = [t.value for t in study.trials if t.value is not None]
ax.plot(range(1, len(scores)+1), np.maximum.accumulate(scores), marker='o')
ax.set(xlabel='Ensayo', ylabel='Mejor accuracy de validación',
       title='Random Search: mejor resultado acumulado')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
best = study.best_params

final_model = LitCNN(
    lr=best["lr"],
    dropout=best["dropout"],
    base_filters=best["base_filters"])

trainer = L.Trainer(
    max_epochs=5,
    callbacks=[L.pytorch.callbacks.EarlyStopping(monitor="val_loss", patience=3, mode="min")],
)
trainer.fit(final_model, train_loader, val_loader)
trainer.test(final_model, test_loader)

